# Judge VQA 
### imports

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from utils.load_env_vars import load_env

load_env()

from torch.utils.data import DataLoader
import json 
import re

from tqdm import tqdm

from judge.query_sglang import query_judge_batched 
from judge.metrics import compute_count_metrics, compute_bool_metrics
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# visualize table existential and universal descriptive questions

### data loading


In [ ]:
from vqa_results_descriptive_utils import load_all_metrics

results = load_all_metrics('universal')  # switch to 'universal' for universal questions

if results:
    print("\nSUMMARY:")
    for dataset, df in results.items():
        print(f"{dataset.upper()}: {len(df)} metric records loaded")
else:
    print("No results loaded!")

### plotting the table

In [ ]:
from vqa_results_descriptive_utils import create_combined_accuracy_table

combined_table, latex_combined = create_combined_accuracy_table(results, datasets_to_use=["unicycle","bicycle"])

### Nightrider results


### Effect of Nightrider on unicycle

In [ ]:
from vqa_results_descriptive_utils import create_nightrider_effect_table

nightrider_comparison, nightrider_diff = create_nightrider_effect_table(results)

### Comparison of All Three Nightrider Splits

Compare nightrider1, nightrider2, and nightrider3 against unicycle, bicycle, and tricycle respectively.

In [ ]:
from vqa_results_descriptive_utils import compare_all_nightrider_splits, generate_all_nightriders_combined_latex

nightrider_comparisons, nightrider_latex, individual_nightrider_latex = compare_all_nightrider_splits(results)

all_nightriders_combined_latex = generate_all_nightriders_combined_latex(nightrider_comparisons)
print(all_nightriders_combined_latex)

### Separate tables for each Nightrider split

In [ ]:
from vqa_results_descriptive_utils import create_individual_nightrider_tables

individual_latex_tables = create_individual_nightrider_tables(results)

In [ ]:
# Check what datasets are loaded
print("Available datasets in results:")
for key in results.keys():
    print(f"  - {key}")

## Accuracy by Tier (Unicycle / Bicycle / Tricycle)

In [ ]:
import matplotlib.pyplot as plt
from vqa_results_descriptive_utils import MODEL_SHORT_NAMES, model_color, build_wide

tiers = ['unicycle', 'bicycle', 'tricycle']
tier_labels = ['Unicycle', 'Bicycle', 'Tricycle']

df_wide = build_wide(results)

fig, ax = plt.subplots(figsize=(8, 5))

for model_label, row in df_wide.iterrows():
    ax.plot(range(len(tiers)), row[tiers].values, marker='o', linewidth=2,
            color=model_color(model_label), label=model_label)

ax.set_xticks(range(len(tiers)))
ax.set_xticklabels(tier_labels, fontsize=12)
ax.set_xlabel('Tier', fontsize=13)
ax.set_ylabel('Mean Accuracy (%)', fontsize=13)
ax.set_title('Model Accuracy Across Dataset Tiers', fontsize=14)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from vqa_results_descriptive_utils import model_color, build_wide

tiers = ['unicycle', 'bicycle', 'tricycle']
tier_labels = ['Unicycle', 'Bicycle', 'Tricycle']

panels = [
    ('attributes', 'Attributes'),
    ('relate',     'Relate'),
    ('compare',    'Compare'),
    (None,         'All (averaged)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

_first_df = next(iter(results.values()))
_qtype = _first_df['question_type'].iloc[0].capitalize()
fig.suptitle(f'Model Accuracy vs. Tier — Scatter + Correlation Line\n({_qtype} questions)',
             fontsize=15, y=1.01)

handles, labels = None, None

for ax, (tmpl, title) in zip(axes, panels):
    df_wide = build_wide(results, template_filter=tmpl)

    records = [
        {'model': m, 'tier_idx': ti, 'accuracy': df_wide.loc[m, tier]}
        for m, row in df_wide.iterrows()
        for ti, tier in enumerate(tiers)
    ]
    df_scatter = pd.DataFrame(records)

    for model_label, grp in df_scatter.groupby('model'):
        ax.scatter(grp['tier_idx'], grp['accuracy'], color=model_color(model_label),
                   s=80, zorder=3, label=model_label)

    x_all = df_scatter['tier_idx'].values
    y_all = df_scatter['accuracy'].values
    slope, intercept, r, p, _ = stats.linregress(x_all, y_all)
    xfit = np.linspace(-0.2, len(tiers) - 0.8, 100)
    p_label = '<0.001' if p < 0.001 else f'{p:.3f}'
    ax.plot(xfit, slope * xfit + intercept, 'k--', linewidth=2, alpha=0.75,
            label=f'Trend  r={r:.2f}, p={p_label}')

    ax.set_xticks(range(len(tiers)))
    ax.set_xticklabels(tier_labels, fontsize=11)
    ax.set_xlabel('Tier', fontsize=11)
    ax.set_ylabel('Mean Accuracy (%)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if handles is None:
        handles, labels = ax.get_legend_handles_labels()

fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.08), frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from vqa_results_descriptive_utils import model_color, build_wide

panels = [
    ('attributes', 'Attributes'),
    ('relate',     'Relate'),
    ('compare',    'Compare'),
    (None,         'All (averaged)'),
]

_first_df = next(iter(results.values()))
_qtype = _first_df['question_type'].iloc[0].capitalize()

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle(
    f'Model Accuracy Across Datasets with Correlation Line\n({_qtype} questions)',
    fontsize=15, y=1.005,
)

x_tiers_base = ['unicycle', 'bicycle', 'tricycle']
handles, labels = None, None

for col_idx, (tmpl, title) in enumerate(panels):

    # ── Row 0: Unicycle → Unicycle Cluttered ─────────────────────────────────
    ax = axes[0, col_idx]
    tiers = ['unicycle', 'unicycle_cluttered']
    tier_labels = ['Unicycle', 'Unicycle Cluttered']

    df_wide = build_wide(results, template_filter=tmpl, tiers=tiers)
    for model_label, row in df_wide.iterrows():
        ax.plot(range(len(tiers)), row[tiers].values, marker='o', linewidth=2,
                color=model_color(model_label), label=model_label)

    if len(df_wide) > 1:
        records = [(ti, df_wide.loc[m, t]) for m in df_wide.index for ti, t in enumerate(tiers)]
        x_all, y_all = zip(*records)
        slope, intercept, r, p, _ = stats.linregress(x_all, y_all)
        xfit = np.linspace(-0.2, len(tiers) - 0.8, 100)
        p_label = '<0.001' if p < 0.001 else f'{p:.3f}'
        ax.plot(xfit, slope * xfit + intercept, 'k--', linewidth=2, alpha=0.7,
                label=f'Trend  r={r:.2f}, p={p_label}')

    ax.set_xticks(range(len(tiers)))
    ax.set_xticklabels(tier_labels, fontsize=10)
    ax.set_xlabel('Dataset', fontsize=11)
    ax.set_ylabel('Mean Accuracy (%)', fontsize=11)
    ax.set_title(f'{title}\nUnicycle → Cluttered', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if handles is None:
        handles, labels = ax.get_legend_handles_labels()

    # ── Row 1: Avg(Uni/Bi/Tri) → Nightrider ──────────────────────────────────
    ax = axes[1, col_idx]
    df_base  = build_wide(results, template_filter=tmpl, tiers=x_tiers_base)
    df_night = build_wide(results, template_filter=tmpl, tiers=['nightrider'])

    common = df_base.index.intersection(df_night.index)
    df_base  = df_base.loc[common]
    df_night = df_night.loc[common]

    x_avg   = df_base.mean(axis=1)
    y_night = df_night['nightrider']
    df_p2   = pd.DataFrame({'Avg(Uni/Bi/Tri)': x_avg, 'Nightrider': y_night})

    for model_label, row in df_p2.iterrows():
        ax.plot([0, 1], row.values, marker='o', linewidth=2,
                color=model_color(model_label), label=model_label)

    if len(common) > 1:
        records2 = [(0, x_avg[m]) for m in common] + [(1, y_night[m]) for m in common]
        x2, y2 = zip(*records2)
        slope2, intercept2, r2, p2, _ = stats.linregress(x2, y2)
        xfit2 = np.linspace(-0.2, 1.2, 100)
        p_label2 = '<0.001' if p2 < 0.001 else f'{p2:.3f}'
        ax.plot(xfit2, slope2 * xfit2 + intercept2, 'k--', linewidth=2, alpha=0.7,
                label=f'Trend  r={r2:.2f}, p={p_label2}')

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Avg(Uni/Bi/Tri)', 'Nightrider'], fontsize=10)
    ax.set_xlabel('Dataset', fontsize=11)
    ax.set_ylabel('Mean Accuracy (%)', fontsize=11)
    ax.set_title(f'{title}\nAvg(Uni/Bi/Tri) → Nightrider', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.06), frameon=True)
plt.tight_layout()
plt.show()
